RAG with webpage data extraction

In [64]:
# !pip install beautifulsoup4 lxml

In [65]:
import os
from pathlib import Path
from dotenv import load_dotenv

cwd = Path.cwd()
env_candidates = [
    cwd / ".env",
    cwd / "notebooks" / ".env",
    cwd.parent / "notebooks" / ".env",
]
env_file = next((p for p in env_candidates if p.exists()), None)
if env_file is None:
    raise FileNotFoundError("No .env found. Expected notebooks/.env with OPENAI_API_KEY.")
load_dotenv(env_file, override=True)

key = (os.getenv("OPENAI_API_KEY") or "").strip()
if not key or "paste_your_key" in key:
    raise ValueError(f"Set OPENAI_API_KEY in {env_file}.")

print(f"Loaded env from: {env_file}")
print("OPENAI_API_KEY configured: True")

Loaded env from: c:\Users\Girish Kulkarni\OneDrive\Documents\LLM_Testing\notebooks\.env
OPENAI_API_KEY configured: True


In [66]:
from langchain_openai import ChatOpenAI

MAX_OUTPUT_TOKENS = 1000

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=MAX_OUTPUT_TOKENS,
)


In [67]:
# Load webpage(s). Put any URL you want to read here.
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://www.descope.com/learn/post/mcp",
]

loader = WebBaseLoader(urls)
documents = loader.load()

print(f"Loaded {len(documents)} webpage(s)")
for doc in documents:
    print(doc.metadata.get("source"), "chars:", len(doc.page_content))

Loaded 1 webpage(s)
https://www.descope.com/learn/post/mcp chars: 27896


In [68]:
# Text Splitting

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)

all_split_docs = text_splitter.split_documents(documents)

len(all_split_docs)  # Total number of chunks after splitting the documents


38

In [69]:
# Embedding the chunks with OpenAI
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_one = embeddings.embed_query(all_split_docs[0].page_content)
vector_two = embeddings.embed_query(all_split_docs[1].page_content)

print(len(vector_one))
print(len(vector_two))

1536
1536


In [70]:
# Vector store (new folder: OpenAI embeddings are 1536-dim, old Ollama index was 768)
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=all_split_docs,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db_openai",
    collection_name="webpage_rag_openai",
)

In [71]:
# Retrieve relevant chunks from the webpage(s)
from langchain_chroma import Chroma

vector_store = Chroma(
    persist_directory="./chroma_langchain_db_openai",
    embedding_function=embeddings,
    collection_name="webpage_rag_openai",
)

question = "What is retrieval-augmented generation?"
retrieved_docs = vector_store.similarity_search(question, k=3)

retrieved_docs

[Document(id='070c0662-6454-4853-befb-b2fbd7ee4942', metadata={'description': 'Learn more about MCP, the open source protocol developed by Anthropic to provide LLMs and AI agents a standardized way to connect with external data sources and tools.', 'language': 'en', 'title': 'What Is the Model Context Protocol (MCP) and How It Works', 'start_index': 4182, 'source': 'https://www.descope.com/learn/post/mcp'}, page_content='about recent data. This requires manually collecting information from various sources, feeding it into the LLM’s chat interface, and then extracting or applying the AI’s output elsewhere.\xa0While several models offer AI-powered web search, and Anthropic’s Claude 3.7 and 4 models boast a Computer Use feature, they still lack direct integration with knowledge stores and tools. Even as major platforms like OpenAI’s ChatGPT and Google’s Gemini add built-in app integrations, these remain platform-specific solutions rather than universal standards.For devs and enterprises, 

In [72]:
# Generate an answer from the retrieved context

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""Answer the question using only the context below.
If the answer is not present in the context, say: I don't know based on the webpage.

Context:
{context}

Question: {question}
Answer:"""

response = llm.invoke(prompt)
print(response.content)

I don't know based on the webpage.


In [73]:
# Retriever over the loaded webpage(s)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

retriever.invoke("What is MCP explain me in 3 tokensa")

[Document(id='e00d53dc-44da-42bc-9b86-a97052d51f91', metadata={'source': 'https://www.descope.com/learn/post/mcp', 'language': 'en', 'start_index': 1771, 'title': 'What Is the Model Context Protocol (MCP) and How It Works', 'description': 'Learn more about MCP, the open source protocol developed by Anthropic to provide LLMs and AI agents a standardized way to connect with external data sources and tools.'}, page_content="MCP standardization is foundational infrastructure for production AI applications. Understanding its architecture is essential for developers building connected AI systems.IdentipediaArrow LeftWhat Is the Model Context Protocol (MCP) and How It WorksJuly 28, 2026Copy linkShare on:Share on LinkedInShare on XShare on BluskyTable of ContentsLLM isolation & the NxM problemOpen table of contentsTable of ContentsLLM isolation & the NxM problemMCP architecture and core componentsHow MCP worksMCP client & server ecosystemSecurity considerations for MCP serversConclusionFAQs ab

In [88]:
# Full RetrievalQA implementation

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Full RetrievalQA implementation
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

question = "What is MCP?"

prompt = ChatPromptTemplate.from_template("""Use only the context below to answer the question.
If the answer is not in the context, say: I don't know based on the documents.
Keep the answer concise and do not add unsupported details.

Context:
{context}

Question: {question}
Answer:""")


def format_documents(documents):
    return "\n\n".join(document.page_content for document in documents)


retrieval_qa_chain = (
    {
        "context": retriever | format_documents,
        "question": lambda value: value,
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = retrieval_qa_chain.invoke(question)
print(answer)


MCP is a lightweight program similar to a microservice that exposes specific capabilities, including tools the model can invoke, resources it can read, and prompts it can use.


In [75]:
# Create multiple set of datasets
# Create goldens from code and push to confidentAi 


test_data = [
    {
        "input": "What is MCP",
        "expected_output": "An MCP server is a lightweight program similar to a microservice that exposes specific capabilities. These include tools the model can invoke, resources it can read, and prompts it can use."
    },
    {
        "input": "Relationship between function calling & Model Context Protocol",
        "expected_output": "The Model Context Protocol (MCP) is an open standard, introduced by Anthropic in November 2024, that gives LLM-based applications (e.g., AI agents) a consistent way to connect with external tools, data sources, and systems."
    },
    {
        "input": "MCP architecture and core components",
        "expected_output": "Similarly, the aim of MCP is to provide a universal way for AI applications to interact with external systems by standardizing context."
    }
]



In [76]:
# Create the goldens
from deepeval.dataset import EvaluationDataset, Golden
goldens = []

for data in test_data:
    golden = Golden(
        input=data["input"],
        expected_output=data["expected_output"]
    )
    goldens.append(golden)

new_dataset = EvaluationDataset(goldens=goldens)
new_dataset

EvaluationDataset(test_cases=[], goldens=[Golden(id=None, input='What is MCP', actual_output=None, expected_output='An MCP server is a lightweight program similar to a microservice that exposes specific capabilities. These include tools the model can invoke, resources it can read, and prompts it can use.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, token_cost=None, input_token_count=None, output_token_count=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(id=None, input='Relationship between function calling & Model Context Protocol', actual_output=None, expected_output='The Model Context Protocol (MCP) is an open standard, introduced by Anthropic in November 2024, that gives LLM-based applications (e.g., AI agents) a consistent way to connect with external tools, data sources, and systems.', context=None, retrieval_context=None, additional_metadat

In [77]:
new_dataset.push("GoldenDataSet")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=4729478;https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/datasets/cmu6azg62000lmh0tfjsbyk9k\https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/datasets/cmu6azg62000lmh0tfjsbyk9k]8;;\

In [86]:
# Add the actual expected result from RAG

# new_dataset.pull(alias="GoldenDataSet")
# pulling is not working as its asking fro active plan of ConfidentAI

<!-- Rettive Data from RAG -->

<!-- # Retrieve data from RAG using existing vector store -->
question = "What is MCP?"

<!-- # Simple retrieval using the existing vector store -->
retrieved_docs = vector_store.similarity_search(question, k=3)

print(f"Retrieved {len(retrieved_docs)} relevant chunk(s)")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content[:800])

<!-- # Build context for the LLM -->
context = "\n\n".join(doc.page_content for doc in retrieved_docs)
print("\n\nContext prepared for LLM.")

In [93]:
question = "What is MCP?"
retrieved_docs = vector_store.similarity_search(question, k=1)

print("Query:", question)
print(f"\nRetrieved {len(retrieved_docs)} result(s)\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"Result {i}:")
    print(doc.page_content.strip())
    print("-" * 80)

Query: What is MCP?

Retrieved 1 result(s)

Result 1:
An MCP server is a lightweight program similar to a microservice that exposes specific capabilities. These include tools the model can invoke, resources it can read, and prompts it can use.
--------------------------------------------------------------------------------


In [80]:
def queryWithContext(question):
    # 1. Get relevant chunks from the vector store
    retrieved_docs = vector_store.similarity_search(question, k=3)

    # 2. Combine retrieved text into one context
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # 3. Ask the model using only that context
    prompt = f"""Answer the question using only the context below.
If the answer is not present in the context, say: I don't know based on the webpage.

Context:
{context}

Question: {question}
Answer:"""

    response = llm.invoke(prompt)
    return {
        "question": question,
        "retrieved_docs": retrieved_docs,
        "answer": response.content
    }    

In [81]:
result = queryWithContext("What is MCP?")
print(result["answer"])

MCP is a lightweight program similar to a microservice that exposes specific capabilities, including tools the model can invoke, resources it can read, and prompts it can use.


In [82]:
from deepeval.dataset import Golden
from deepeval.test_case import LLMTestCase
from typing import List


def convertsGoldenInToTestcases(goldens: List[Golden]) -> List[LLMTestCase]:
    test_cases = []

    for golden in goldens:
        result = queryWithContext(golden.input)

        context = "\n\n".join(doc.page_content for doc in result["retrieved_docs"])
        rag_response = result["answer"]

        test_case = LLMTestCase(
            input=golden.input,
            actual_output=rag_response,
            expected_output=golden.expected_output,
            retrieval_context=[context]
        )

        test_cases.append(test_case)

    return test_cases


test_cases = convertsGoldenInToTestcases(new_dataset.goldens)
print(len(test_cases))

3


In [83]:
from deepeval.dataset import Golden
from deepeval.test_case import LLMTestCase
from typing import List


def convertsGoldenInToTestcases(goldens: List[Golden]) -> List[LLMTestCase]:
    test_cases = []

    for golden in goldens:
        result = queryWithContext(golden.input)

        # convert retrieved docs into one context string
        context = "\n\n".join(doc.page_content for doc in result["retrieved_docs"])
        rag_response = result["answer"]

        test_case = LLMTestCase(
            input=golden.input,
            actual_output=rag_response,
            expected_output=golden.expected_output,
            retrieval_context=[context]
        )

        test_cases.append(test_case)

    return test_cases


test_cases = convertsGoldenInToTestcases(new_dataset.goldens)
test_cases
print(len(test_cases))

3


In [84]:
from deepeval.dataset import Golden
from typing import List

def validate_rag_against_goldens(goldens: List[Golden]):
    results = []

    for golden in goldens:
        rag_result = queryWithContext(golden.input)

        retrieved_text = "\n\n".join(doc.page_content for doc in rag_result["retrieved_docs"])
        answer = rag_result["answer"]

        expected = golden.expected_output.lower()
        retrieved = retrieved_text.lower()

        # simple check: does expected content appear in retrieved context or answer
        passed = expected in retrieved or expected in answer.lower()

        results.append({
            "input": golden.input,
            "passed": passed,
            "retrieved_context": retrieved_text[:800],
            "answer": answer,
        })

    return results


validation_results = validate_rag_against_goldens(new_dataset.goldens)
validation_results

[{'input': 'What is MCP',
  'passed': True,
  'retrieved_context': 'An MCP server is a lightweight program similar to a microservice that exposes specific capabilities. These include tools the model can invoke, resources it can read, and prompts it can use.\n\nAn MCP server is a lightweight program similar to a microservice that exposes specific capabilities. These include tools the model can invoke, resources it can read, and prompts it can use.\n\nAn MCP server is a lightweight program similar to a microservice that exposes specific capabilities. These include tools the model can invoke, resources it can read, and prompts it can use.',
  'answer': 'MCP is a lightweight program similar to a microservice that exposes specific capabilities, including tools the model can invoke, resources it can read, and prompts it can use.'},
 {'input': 'Relationship between function calling & Model Context Protocol',
  'passed': True,
  'retrieved_context': 'Context Protocol (MCP)?The Model Context Pr

In [85]:
passed_count = sum(1 for x in validation_results if x["passed"])
total_count = len(validation_results)

print(f"Passed: {passed_count}/{total_count}")
for r in validation_results:
    print(f"\nQuestion: {r['input']}")
    print("Passed:", r["passed"])

Passed: 3/3

Question: What is MCP
Passed: True

Question: Relationship between function calling & Model Context Protocol
Passed: True

Question: MCP architecture and core components
Passed: True
